# 04g — XGBoost, LightGBM et Random Forest

Comparaison simple, sur les **mêmes 31 features** que le 04f et les mêmes plis par moteur. Chaque modèle a ses hyperparamètres réglés par Optuna (60 essais pour XGBoost au 04f, 40 pour les deux autres, réglés sur les mêmes plis de recherche).

Résultat : LightGBM fait jeu égal avec XGBoost, Random Forest est nettement moins bon. Aucun ne justifie de changer le modèle final du 04f.


In [ ]:
import sys
sys.path.insert(0, "..")  # pour importer src/ depuis notebooks/

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    max_error, mean_absolute_error, median_absolute_error, r2_score,
)
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5         # fenetre de lissage (Phase 1)
PLAFOND_RUL = 125   # plafond de la RUL (Phase 2)
K_INIT = 20         # cycles servant a definir l'etat initial d'un moteur (04e)
GRAINE = 42

# --- train : capteurs constants retires, min-max, lissage, RUL plafonnee (memes etapes que la Phase 3) ---
train = load_fd001("train")
capteurs_constants = find_constant_sensors(train, get_sensor_columns(train.columns))
train = train.drop(columns=capteurs_constants)
capteurs = get_sensor_columns(train.columns)  # les 15 capteurs utiles
train, scaler = normalize_minmax(train, capteurs)
train = lisser_capteurs(train, capteurs, fenetre=FENETRE)
train = calculer_rul(train)
train = plafonner_rul(train, plafond=PLAFOND_RUL)


def ecart_etat_initial(df, capteurs, k):
    """
    Pour chaque capteur : valeur lissee du cycle courant moins la moyenne des k premiers cycles du moteur.

    Parameters
    ----------
    df : pandas.DataFrame
        Tableau trie par moteur puis par cycle, capteurs deja lisses.
    capteurs : list de str
        Noms des capteurs.
    k : int
        Nombre de cycles de reference ; avant le cycle k, moyenne des cycles connus.

    Returns
    -------
    pandas.DataFrame
        Une colonne "<capteur>|ecartinit_<k>" par capteur, meme index que df.
    """
    unites = df["unit_number"].values
    colonnes = {}
    for capteur in capteurs:
        colonnes[f"{capteur}|ecartinit_{k}"] = np.zeros(len(df))  # espace reserve
    for moteur_id in np.unique(unites):  # un moteur a la fois : jamais de melange entre moteurs
        positions = np.where(unites == moteur_id)[0]
        for capteur in capteurs:
            valeurs = df[capteur].values[positions]
            somme_cumulee = np.cumsum(valeurs)
            moyenne_debut = somme_cumulee / np.arange(1, len(valeurs) + 1)  # cycle < k : moyenne des cycles connus
            if len(valeurs) >= k:
                moyenne_debut[k:] = somme_cumulee[k - 1] / k                # cycle >= k : moyenne figee des k premiers
            colonnes[f"{capteur}|ecartinit_{k}"][positions] = valeurs - moyenne_debut
    return pd.DataFrame(colonnes, index=df.index)


def features_31(df):
    """Jeu retenu au 04e : 15 capteurs lisses, time_cycles, 15 ecarts a l'etat initial."""
    return pd.concat([df[capteurs + ["time_cycles"]], ecart_etat_initial(df, capteurs, K_INIT)], axis=1)


X31 = features_31(train)
assert X31.shape[1] == 31
y = train["RUL_cappee"]
groups = train["unit_number"]
plis = list(GroupKFold(n_splits=5, shuffle=True, random_state=GRAINE).split(X31, y, groups=groups))  # 5 plis par moteur

print(f"{X31.shape[1]} features, 5 plis par moteur.")


In [ ]:
# --- Comparaison : validation sur 5 decoupages par moteur neufs, puis test lu une seule fois par modele ---
def nasa_moyen(y_true, y_pred):
    """Score NASA moyen par prediction : surestimer (ecart > 0) coute plus cher que sous-estimer."""
    ecart = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.mean(np.where(ecart < 0, np.exp(-ecart / 13) - 1, np.exp(ecart / 10) - 1)))


def metriques(y_true, y_pred):
    """Metriques de regression du notebook 04a (celles gardees)."""
    ecarts = y_pred - y_true  # positif = surestimation
    return {
        "RMSE": float(np.sqrt(np.mean(ecarts ** 2))), "MAE": mean_absolute_error(y_true, y_pred), "R2": r2_score(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred), "MaxErr": max_error(y_true, y_pred), "Biais": float(np.mean(ecarts)),
        "NASA_moy": nasa_moyen(y_true, y_pred),
    }


# hyperparametres issus des recherches Optuna (XGBoost : 04f ; LightGBM et Random Forest : 40 essais, TPE graine 42, plis du 04f)
MODELES = {
    "XGBoost": lambda: XGBRegressor(
        random_state=GRAINE, max_depth=4, learning_rate=0.0633, n_estimators=422, subsample=0.5789,
        colsample_bytree=0.8676, min_child_weight=1.3287, reg_lambda=29.1205),
    "LightGBM": lambda: LGBMRegressor(
        random_state=GRAINE, verbose=-1, n_estimators=549, learning_rate=0.02744, num_leaves=8, min_child_samples=72,
        subsample=0.6534, subsample_freq=1, colsample_bytree=0.9839, reg_lambda=2.5794),
    "Random Forest": lambda: RandomForestRegressor(
        random_state=GRAINE, n_jobs=-1, n_estimators=391, max_depth=18, min_samples_leaf=3, max_features=0.5432),
}

# --- validation : memes plis pour les 3 modeles, 5 graines jamais utilisees pour la recherche ---
rmse_val = {nom: [] for nom in MODELES}
nasa_val = {nom: [] for nom in MODELES}
for graine in range(1000, 1005):
    plis_neufs = list(GroupKFold(n_splits=5, shuffle=True, random_state=graine).split(X31, y, groups=groups))
    for nom, fabrique in MODELES.items():
        erreurs_carre, nasas = [], []
        for idx_train, idx_val in plis_neufs:
            pred = fabrique().fit(X31.iloc[idx_train], y.iloc[idx_train]).predict(X31.iloc[idx_val])
            y_val = y.iloc[idx_val].values
            erreurs_carre.append(np.mean((y_val - pred) ** 2))
            nasas.append(nasa_moyen(y_val, pred))
        rmse_val[nom].append(np.sqrt(np.mean(erreurs_carre)))
        nasa_val[nom].append(np.mean(nasas))

# --- test officiel : dernier cycle de chaque moteur, vraie RUL dans RUL_FD001.txt ---
test = load_fd001("test").drop(columns=capteurs_constants)
test[capteurs] = scaler.transform(test[capteurs])  # transform seul : scaler appris sur le train
test = lisser_capteurs(test, capteurs, fenetre=FENETRE)
derniers = test.groupby("unit_number")["time_cycles"].idxmax().sort_index().values
y_test = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None)[0].values.astype(float)
X_test = features_31(test).loc[derniers]

lignes = []
predictions_test = {}
for nom, fabrique in MODELES.items():
    pred_test = np.clip(fabrique().fit(X31, y).predict(X_test), 0, None)  # RUL negative sans sens physique
    predictions_test[nom] = pred_test
    ecart_rmse = np.array(rmse_val[nom]) - np.array(rmse_val["XGBoost"])
    lignes.append({
        "modele": nom, "RMSE val": np.mean(rmse_val[nom]), "NASA moy val": np.mean(nasa_val[nom]),
        "ecart RMSE vs XGBoost": ecart_rmse.mean(), "plus bas que XGBoost": f"{int((ecart_rmse < 0).sum())}/5",
        **{f"{m} test": v for m, v in metriques(y_test, pred_test).items()},
    })
pd.set_option("display.width", 250)
print(pd.DataFrame(lignes).round(3).to_string(index=False))


In [ ]:
# --- Graphiques : les 3 modeles cote a cote ---
import matplotlib.pyplot as plt

noms = list(MODELES)
couleurs = ["tab:blue", "tab:green", "tab:orange"]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

# 1. RMSE : validation (moyenne des 5 graines, barre = min-max) et test
ax = axes[0]
x = np.arange(len(noms))
moy_val = [np.mean(rmse_val[n]) for n in noms]
barres = [[moy_val[i] - min(rmse_val[n]) for i, n in enumerate(noms)], [max(rmse_val[n]) - moy_val[i] for i, n in enumerate(noms)]]
rmse_test = [np.sqrt(np.mean((predictions_test[n] - y_test) ** 2)) for n in noms]
ax.bar(x - 0.2, moy_val, 0.4, yerr=barres, capsize=4, label="validation (5 graines)")
ax.bar(x + 0.2, rmse_test, 0.4, label="test (100 moteurs)")
for i in range(len(noms)):
    ax.text(x[i] - 0.2, moy_val[i] + 0.15, f"{moy_val[i]:.2f}", ha="center", fontsize=9)
    ax.text(x[i] + 0.2, rmse_test[i] + 0.15, f"{rmse_test[i]:.2f}", ha="center", fontsize=9)
ax.set_xticks(x, noms)
ax.set_ylabel("RMSE (cycles)")
ax.set_title("Erreur de prediction (plus bas = mieux)")
ax.set_ylim(0, 16)  # place pour la legende
ax.legend(fontsize=8, loc="upper left", ncol=2)

# 2. distribution des ecarts (predit - vrai) sur le test
ax = axes[1]
boite = ax.boxplot([predictions_test[n] - y_test for n in noms], tick_labels=noms, patch_artist=True)
for patch, couleur in zip(boite["boxes"], couleurs):
    patch.set_facecolor(couleur)
    patch.set_alpha(0.5)
ax.axhline(0, color="k", linewidth=1, linestyle="--")
ax.set_ylabel("RUL predite - RUL vraie (cycles)")
ax.set_title("Ecarts sur le test (> 0 = surestimation)")

# 3. predit contre vrai sur le test
ax = axes[2]
for n, couleur in zip(noms, couleurs):
    ax.scatter(y_test, predictions_test[n], s=14, alpha=0.5, color=couleur, label=n)
ax.plot([0, 150], [0, 150], "k--", linewidth=1)
ax.set_xlabel("RUL vraie (cycles)")
ax.set_ylabel("RUL predite (cycles)")
ax.set_title("Test : 100 moteurs")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
